### Import our dependencies and initialize our llm client

In [1]:
import openai
from langsmith import Client
from langsmith.wrappers import wrap_openai
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
openai_client = wrap_openai(openai.Client())

### Set up our redaction fn, referencing these [docs](https://docs.smith.langchain.com/observability/how_to_guides/mask_inputs_outputs)

In [3]:
def redact_system_messages(inputs: dict) -> dict:
    """Redact system messages from the inputs."""
    messages = inputs.get("messages", [])
    redacted = [
        {"role": m.get("role"),
         "content": "REDACTED"}
        if m.get("role") == "system"
        else m
        for m in messages
    ]
    return {**inputs, "messages": redacted}

### Define our LangSmith Client with the redaction function we defined above

In [4]:
langsmith_client = Client(
  hide_inputs=lambda inputs: redact_system_messages(inputs)
  # Optionally we could set hide_inputs=lambda inputs: {}, hide_outputs=lambda outputs: {} to mask all inputs and outputs
)

### Make model call and pass our LangSmith client in the `langsmith_extra` field

In [5]:

openai_client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[
      {"role": "system", "content": "You are a helpful assistant."},
      {"role": "user", "content": "Hello, how are you!"},
  ],
  langsmith_extra={"client": langsmith_client},
)

ChatCompletion(id='chatcmpl-Bp0sTWbMq9bhXEwWXzI3nI45eKklA', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Hello! I'm here and ready to help you. How can I assist you today?", refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1751497801, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_34a54ae93c', usage=CompletionUsage(completion_tokens=17, prompt_tokens=23, total_tokens=40, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

We should see the system prompt gets redacted. You can see a sample trace [here](https://smith.langchain.com/public/7602d303-8f1a-41b3-beae-6fab07be3fa5/r)